In [ ]:
def plot_pitcher_dashboard(player_name, start_date, end_date, pitch_types):
    from pybaseball import statcast_pitcher, playerid_lookup
    import pandas as pd
    import matplotlib.pyplot as plt

    # player ID
    df = playerid_lookup(player_name.split()[1], player_name.split()[0])
    
    if df.empty:
        print(f"No player found with name {player_name}")
        return
    
    player_id = df['key_mlbam'].iloc[0]

    # data
    pitcher_df = statcast_pitcher(start_date, end_date, player_id)
    pitcher_df["game_date"] = pd.to_datetime(pitcher_df["game_date"])

    # create full date range (with FIXED X-AXIS)
    full_dates = pd.date_range(start=start_date, end=end_date)
    full_df = pd.DataFrame({"game_date": full_dates})

    plt.figure(figsize=(12,6))

    for pitch_type in pitch_types:
        pitch_df = pitcher_df[pitcher_df["pitch_type"] == pitch_type].copy()

        if pitch_df.empty:
            print(f"No data for {pitch_type}")
            continue

        # clean data
        pitch_df = pitch_df.rename(columns={"release_speed": "velo"})
        pitch_df["velo"] = pd.to_numeric(pitch_df["velo"], errors="coerce")
        pitch_df = pitch_df.dropna(subset=["velo"])

        # group by game date
        game_avg = pitch_df.groupby("game_date")["velo"].mean().reset_index()

        # merge with full date range
        game_avg = pd.merge(full_df, game_avg, on="game_date", how="left")

        # interpolate missing values
        game_avg["velo_interp"] = game_avg["velo"].interpolate()

        # rolling average (smoother trend)
        game_avg["velo_rolling"] = game_avg["velo_interp"].rolling(5, min_periods=1).mean()

        # plot raw (faded)
        plt.plot(
            game_avg["game_date"],
            game_avg["velo"],
            alpha=0.25,
            linestyle="--"
        )

        # plot rolling average (main line)
        plt.plot(
            game_avg["game_date"],
            game_avg["velo_rolling"],
            label=pitch_type
        )

    # graph style
    plt.xlabel("Date")
    plt.ylabel("Velocity (mph)")
    plt.title(f"{player_name} Pitch Velocity Trends")
    plt.legend(title="Pitch Type")
    plt.grid(True)

    plt.show()

In [ ]:
plot_pitcher_dashboard(
    player_name="Gerrit Cole",
    start_date="2024-04-01",
    end_date="2024-09-30",
    pitch_types=["FF", "SL", "CH"]
)